# Άσκηση 1: Εξερεύνηση Δεδομένων — Τιμές Κατοικιών
## Το Σενάριο (Business Problem)

Είστε αναλυτής δεδομένων σε εταιρεία Real Estate. Το πρόβλημα που καλείστε να λύσετε είναι η **εκτίμηση της τιμής πώλησης κατοικιών** (`SalePrice`) με βάση τα χαρακτηριστικά τους.

**Dataset:** [House Prices — Kaggle](https://www.kaggle.com/datasets/juhibhojani/house-price)

> Το dataset περιέχει **1.460 κατοικίες** από το Ames, Iowa (ΗΠΑ) με **80 χαρακτηριστικά**: από την επιφάνεια και το έτος κατασκευής, μέχρι την ποιότητα εξωτερικού χώρου και τον τύπο θέρμανσης.

Στόχος αυτής της άσκησης είναι να εφαρμόσουμε τη φάση **Intelligence** του μοντέλου Simon:
1. **Intelligence:** Θα φορτώσουμε το dataset, θα εξετάσουμε τη δομή του και θα εντοπίσουμε προβλήματα (ελλείπουσες τιμές, ακραίες τιμές, κατανομές).
2. **Design:** Θα αναλύσουμε τις συσχετίσεις — ποια χαρακτηριστικά επηρεάζουν περισσότερο την τιμή πώλησης;

> **Οδηγία:** Σε κάθε κελί κώδικα που περιέχει `# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ`, αντικαταστήστε το σχόλιο με τον κατάλληλο κώδικα.

## Βασικές Έννοιες για αυτή την Άσκηση

Πριν ξεκινήσουμε, παρουσιάζουμε τις κύριες έννοιες που θα συναντήσουμε.

---

### 1. Τύποι Δεδομένων (Data Types)

| Τύπος | Περιγραφή | Παραδείγματα στο dataset |
|---|---|---|
| **Αριθμητικά — Συνεχή** | Αριθμοί με ενδιάμεσες τιμές | `SalePrice`, `GrLivArea`, `LotArea` |
| **Αριθμητικά — Διακριτά** | Ακέραιοι αριθμοί που μετράνε κάτι | `BedroomAbvGr`, `FullBath`, `GarageCars` |
| **Κατηγορικά — Ονομαστικά** | Κατηγορίες χωρίς φυσική σειρά | `Neighborhood`, `HouseStyle`, `SaleType` |
| **Κατηγορικά — Τακτικά** | Κατηγορίες με σειρά/βαθμίδες | `ExterQual` (Ex/Gd/TA/Fa/Po), `KitchenQual` |

---

### 2. Ελλείπουσες Τιμές (Missing Values)

| Έννοια | Σημασία |
|---|---|
| **NaN** (Not a Number) | Η τιμή λείπει εντελώς από το dataset |
| **MCAR** | Missing Completely At Random — τυχαία απουσία |
| **MAR** | Missing At Random — σχετίζεται με άλλες στήλες |
| **MNAR** | Missing Not At Random — η ίδια η τιμή εξηγεί γιατί λείπει |

> **Πρακτική σημείωση:** Ορισμένες στήλες όπως `PoolQC`, `Alley`, `MiscFeature` έχουν πολλά `NaN` γιατί το σπίτι **δεν έχει** το χαρακτηριστικό — δεν είναι "λάθος" αλλά "δεν υπάρχει".

---

### 3. Κατανομές (Distributions)

| Κατανομή | Χαρακτηριστικά | Τι σημαίνει |
|---|---|---|
| **Κανονική (Normal / Gaussian)** | Συμμετρική, καμπανόσχημη | Οι περισσότερες τιμές είναι κοντά στον μέσο |
| **Δεξιά Ασύμμετρη (Right Skewed)** | Μακριά "ουρά" δεξιά | Λίγες πολύ υψηλές τιμές "τραβούν" τον μέσο προς τα δεξιά |
| **Log-Normal** | Με λογαριθμική κλίμακα γίνεται κανονική | Τυπικό για τιμές (SalePrice), εισοδήματα, κ.λπ. |

> 📊 Ο `SalePrice` συνήθως ακολουθεί **log-normal κατανομή** — η διεθμική `log(SalePrice)` είναι κανονική.

---

### 4. Συσχέτιση Pearson (Pearson Correlation)

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i-\bar{x})^2 \cdot \sum(y_i-\bar{y})^2}} \quad \in [-1, 1]$$

| Τιμή r | Ερμηνεία |
|---|---|
| $r > 0.7$ | Ισχυρή θετική συσχέτιση |
| $0.4 < r \leq 0.7$ | Μέτρια θετική συσχέτιση |
| $0 < r \leq 0.4$ | Ασθενής θετική συσχέτιση |
| $r < 0$ | Αρνητική συσχέτιση |

> ⚠️ **Η συσχέτιση μετράει μόνο γραμμικές σχέσεις.** Δύο μεταβλητές μπορεί να σχετίζονται έντονα (π.χ. τετραγωνική σχέση) αλλά να έχουν r ≈ 0.

---
## Βήμα 1: Εγκατάσταση Βιβλιοθηκών

Εκτελέστε το παρακάτω κελί για να εγκαταστήσετε τις απαραίτητες βιβλιοθήκες (αν δεν είναι ήδη εγκατεστημένες).

In [ ]:
pip install pandas matplotlib seaborn numpy scipy

---
## Βήμα 2: Εισαγωγή Βιβλιοθηκών

Εισάγετε τις βιβλιοθήκες που θα χρειαστείτε για την ανάλυση.

**Βοήθεια:** Χρησιμοποιήστε τις συμβάσεις:  
`import pandas as pd`, `import matplotlib.pyplot as plt`, `import seaborn as sns`, `import numpy as np`

In [ ]:
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# Εισαγωγή βιβλιοθηκών
# import ...

# Ρυθμίσεις εμφάνισης
# pd.set_option(...)
# plt.rcParams[...]
# sns.set_theme(...)

---
## Βήμα 3: Φόρτωση Δεδομένων

Κατεβάστε το αρχείο `train.csv` από το [Kaggle](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data) και τοποθετήστε το στον ίδιο φάκελο με αυτό το notebook.

**Ερώτηση:** Πόσες γραμμές και στήλες έχει το dataset;

In [ ]:
# Φόρτωση του dataset
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# df = pd.read_csv('train.csv')
# print(f"Σχήμα dataset: {df.shape}")
# print(f"Γραμμές: {df.shape[0]}, Στήλες: {df.shape[1]}")

---
## Βήμα 4: Πρώτη Ματιά στα Δεδομένα

### 4α. Εμφάνιση των πρώτων εγγραφών

Χρησιμοποιήστε τη μέθοδο `.head()` για να δείτε τις πρώτες 5 εγγραφές.

In [ ]:
# Εμφάνιση των πρώτων 5 εγγραφών
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

### 4β. Πληροφορίες για τις στήλες

Χρησιμοποιήστε τη μέθοδο `.info()` για να δείτε τους τύπους δεδομένων και τον αριθμό μη-κενών τιμών ανά στήλη.

**Ερώτηση:** Πόσες στήλες είναι αριθμητικές (`int64`, `float64`) και πόσες κατηγορικές (`object`);

In [ ]:
# Πληροφορίες για τύπους δεδομένων
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

### 4γ. Βασικά Στατιστικά

Χρησιμοποιήστε τη μέθοδο `.describe()` για να δείτε τα βασικά στατιστικά (μέσος, τυπική απόκλιση, ελάχιστο, τεταρτημόρια, μέγιστο) για κάθε αριθμητική στήλη.

**Ερώτηση:** Ποιο είναι το ελάχιστο και μέγιστο `SalePrice`; Πόση είναι η διαφορά τους;

In [ ]:
# Βασικά στατιστικά για αριθμητικές στήλες
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

---
## Βήμα 5: Ανάλυση Ελλειπουσών Τιμών (Missing Values)

Ένα σημαντικό βήμα σε κάθε ανάλυση δεδομένων είναι ο εντοπισμός και η κατανόηση των ελλειπουσών τιμών.

**Ερώτηση:** Ποιες στήλες έχουν τα περισσότερα `NaN`; Ποιο ποσοστό αντιπροσωπεύουν;

In [ ]:
# Υπολογισμός ελλειπουσών τιμών ανά στήλη
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# Βήμα 1: Μετρήστε τα NaN ανά στήλη (isnull().sum())
# missing_count = ...

# Βήμα 2: Υπολογίστε ποσοστό (/ len(df) * 100)
# missing_pct = ...

# Βήμα 3: Δημιουργήστε DataFrame με τα αποτελέσματα
# missing_df = pd.DataFrame({'count': missing_count, 'percentage': missing_pct})

# Βήμα 4: Κρατήστε μόνο τις στήλες με NaN > 0 και ταξινομήστε φθίνουσα
# missing_df = missing_df[missing_df['count'] > 0].sort_values('percentage', ascending=False)

# print(f"Στήλες με ελλείπουσες τιμές: {len(missing_df)}")
# missing_df

### Οπτικοποίηση Ελλειπουσών Τιμών

Δημιουργήστε οριζόντιο ραβδόγραμμα (horizontal bar chart) που να δείχνει το **ποσοστό ελλειπουσών τιμών** για τις στήλες με NaN > 0.

**Βοήθεια:** Χρησιμοποιήστε `ax.barh()` με τα δεδομένα από το `missing_df` που φτιάξατε παραπάνω.

In [ ]:
# Οριζόντιο ραβδόγραμμα ελλειπουσών τιμών
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# fig, ax = plt.subplots(figsize=(10, 8))
# ax.barh(missing_df.index, missing_df['percentage'], color='steelblue')
# ax.axvline(x=50, color='red', linestyle='--', label='50% όριο')
# ax.set_xlabel('Ποσοστό Ελλειπουσών Τιμών (%)')
# ax.set_title('Ελλείπουσες Τιμές ανά Στήλη')
# ax.legend()
# plt.tight_layout()
# plt.show()

---
## Βήμα 6: Κατανομή της Τιμής Πώλησης (SalePrice)

Η μεταβλητή-στόχος `SalePrice` είναι το κέντρο της ανάλυσής μας. Πρέπει να καταλάβουμε την κατανομή της.

**Θεωρία:** Πολλές τιμές σπιτιών ακολουθούν **log-normal κατανομή**: στη γραμμική κλίμακα είναι ασύμμετρες (δεξί "skew"), αλλά αν εφαρμόσουμε λογαριθμική μετατροπή `log(SalePrice)`, η κατανομή γίνεται κανονική.

**Ερώτηση:** Είναι ο `SalePrice` ασύμμετρος; Επαληθεύεται στο dataset μας;

In [ ]:
# Κατανομή SalePrice: Ιστόγραμμα + Boxplot (γραμμική & λογαριθμική κλίμακα)
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle('Κατανομή SalePrice', fontsize=14, fontweight='bold')

# --- Γραμμική κλίμακα ---
# axes[0, 0]: Ιστόγραμμα SalePrice (bins=50, color='steelblue')
# Προσθέστε κατακόρυφες γραμμές για μέσο (mean) και διάμεσο (median)

# axes[0, 1]: Boxplot SalePrice
# Με υπολογισμό και εκτύπωση IQR και αριθμού outliers (< Q1-1.5*IQR ή > Q3+1.5*IQR)

# --- Λογαριθμική κλίμακα ---
# Δημιουργήστε log_price = np.log(df['SalePrice'])
# axes[1, 0]: Ιστόγραμμα log(SalePrice) (bins=50, color='darkorange')
# axes[1, 1]: Boxplot log(SalePrice)

# plt.tight_layout()
# plt.show()

# Εκτύπωση στατιστικών
# print(f"SalePrice — μέσος: {df['SalePrice'].mean():.0f}€, διάμεσος: {df['SalePrice'].median():.0f}€")
# print(f"Ασυμμετρία (skewness): {df['SalePrice'].skew():.3f}")

---
## Βήμα 7: Αριθμητικές vs Κατηγορικές Στήλες

### 7α. Διαχωρισμός στηλών

Χωρίστε τις στήλες σε αριθμητικές και κατηγορικές. Χρησιμοποιήστε `df.select_dtypes()`.

**Βοήθεια:** `df.select_dtypes(include='number')` για αριθμητικές, `df.select_dtypes(include='object')` για κατηγορικές.

In [ ]:
# Διαχωρισμός αριθμητικών / κατηγορικών στηλών
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# num_cols = df.select_dtypes(include='number').columns.tolist()
# cat_cols = df.select_dtypes(include='object').columns.tolist()

# print(f"Αριθμητικές στήλες: {len(num_cols)}")
# print(f"Κατηγορικές στήλες: {len(cat_cols)}")

### 7β. Συχνότητα Κατηγορικών Μεταβλητών

Επιλέξτε **3 κατηγορικές στήλες** που θεωρείτε σημαντικές (π.χ. `Neighborhood`, `HouseStyle`, `SaleCondition`) και δημιουργήστε **ραβδογράμματα συχνότητας** για κάθε μία.

**Ερώτηση:** Ποια γειτονιά (`Neighborhood`) εμφανίζεται πιο συχνά στο dataset;

In [ ]:
# Ραβδογράμματα συχνότητας για 3 κατηγορικές μεταβλητές
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# cols_to_plot = ['Neighborhood', 'HouseStyle', 'SaleCondition']

# fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# fig.suptitle('Συχνότητα Κατηγορικών Μεταβλητών', fontsize=14, fontweight='bold')

# for ax, col in zip(axes, cols_to_plot):
#     counts = df[col].value_counts()
#     ax.bar(counts.index, counts.values, color='steelblue')
#     ax.set_title(col)
#     ax.set_xlabel('')
#     ax.set_ylabel('Αριθμός κατοικιών')
#     ax.tick_params(axis='x', rotation=45)

# plt.tight_layout()
# plt.show()

---
## Βήμα 8: Συσχέτιση με την Τιμή Πώλησης (SalePrice)

### 8α. Υπολογισμός Συσχετίσεων

Υπολογίστε τον **συντελεστή συσχέτισης Pearson** μεταξύ κάθε αριθμητικής στήλης και του `SalePrice`.

**Βοήθεια:** `df.corr()['SalePrice'].sort_values(ascending=False)`

**Ερώτηση:** Ποιες είναι οι **5 πιο θετικά** και οι **5 πιο αρνητικά** συσχετισμένες μεταβλητές;

In [ ]:
# Συσχέτιση αριθμητικών στηλών με SalePrice
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# corr_with_price = df[num_cols].corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

# print("Top 10 θετικά συσχετισμένες:")
# print(corr_with_price.head(10).to_string())
# print("\nTop 5 αρνητικά συσχετισμένες:")
# print(corr_with_price.tail(5).to_string())

### 8β. Heatmap Συσχετίσεων (Top 10 μεταβλητές)

Δημιουργήστε **heatmap** με τις 10 πιο συσχετισμένες μεταβλητές με τον `SalePrice`.

**Βοήθεια:** `sns.heatmap(df[top10_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')`

In [ ]:
# Heatmap για τις top 10 + SalePrice
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# top10_features = corr_with_price.head(10).index.tolist()
# top10_cols = top10_features + ['SalePrice']

# fig, ax = plt.subplots(figsize=(12, 9))
# corr_matrix = df[top10_cols].corr()
# sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
#             vmin=-1, vmax=1, center=0,
#             square=True, linewidths=0.5, ax=ax)
# ax.set_title('Heatmap Συσχετίσεων — Top 10 Μεταβλητές + SalePrice', fontsize=13)
# plt.tight_layout()
# plt.show()

### 8γ. Scatter Plots — Top 5 Μεταβλητές vs SalePrice

Δημιουργήστε **scatter plots** για τις 5 πιο συσχετισμένες μεταβλητές με τον `SalePrice`.

**Βοήθεια:** `ax.scatter(df[col], df['SalePrice'], alpha=0.4, s=10)`

**Ερώτηση:** Βλέπετε γραμμική σχέση σε όλες; Υπάρχουν outliers; Ποια μεταβλητή δείχνει την πιο "καθαρή" σχέση;

In [ ]:
# Scatter plots: Top 5 μεταβλητές vs SalePrice
# ΕΔΩ Ο ΚΩΔΙΚΑΣ ΣΑΣ

# top5_features = corr_with_price.head(5).index.tolist()

# fig, axes = plt.subplots(1, 5, figsize=(18, 4))
# fig.suptitle('Top 5 Μεταβλητές vs SalePrice', fontsize=14, fontweight='bold')

# for ax, col in zip(axes, top5_features):
#     ax.scatter(df[col], df['SalePrice'], alpha=0.4, s=10, color='steelblue')
#     r = df[[col, 'SalePrice']].corr().iloc[0, 1]
#     ax.set_title(f'{col}\nr={r:.2f}', fontsize=9)
#     ax.set_xlabel(col, fontsize=8)
#     ax.set_ylabel('SalePrice' if ax == axes[0] else '')

# plt.tight_layout()
# plt.show()

---
## Σύνοψη — Φάση Intelligence (Simon Model)

Συμπληρώστε τον παρακάτω πίνακα με τα αποτελέσματα της ανάλυσής σας:

| Ερώτηση | Απάντηση |
|---|---|
| Πόσες γραμμές / στήλες έχει το dataset; | |
| Πόσες αριθμητικές στήλες; | |
| Πόσες κατηγορικές στήλες; | |
| Πόσες στήλες έχουν ελλείπουσες τιμές; | |
| Ποια στήλη έχει τα περισσότερα NaN; | |
| Είναι ο SalePrice ασύμμετρος (skewed); | |
| Ποια μεταβλητή έχει τη μεγαλύτερη συσχέτιση με το SalePrice; | |
| Τιμή της συσχέτισης αυτής; | |

---

## Bonus: Προαιρετικές Ασκήσεις

### Bonus 1: Ανάλυση SalePrice ανά Γειτονιά
Φτιάξτε boxplot για τον `SalePrice` ανά `Neighborhood`. Ποιες γειτονιές έχουν τις υψηλότερες τιμές;

```python
# fig, ax = plt.subplots(figsize=(14, 6))
# order = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index
# df.boxplot(column='SalePrice', by='Neighborhood', ax=ax, rot=90, figsize=(14, 6))
# plt.tight_layout()
# plt.show()
```

### Bonus 2: Σχέση Επιφάνειας και Τιμής
Εξετάστε τη σχέση `GrLivArea` (επιφάνεια πάνω από το ισόγειο) με το `SalePrice`. Υπάρχουν outliers με μεγάλη επιφάνεια αλλά χαμηλή τιμή;

### Bonus 3: Ποιότητα vs Τιμή
Η στήλη `OverallQual` (Συνολική Ποιότητα, κλίμακα 1–10) έχει υψηλή συσχέτιση με το `SalePrice`. Φτιάξτε ένα boxplot `SalePrice` ανά `OverallQual` για να δείτε πώς αυξάνεται η τιμή με την ποιότητα.